In [7]:
import numpy as np
import pandas as pd
import os
import datetime as dt
from sklearn.linear_model import LogisticRegression, ElasticNet, LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (mean_squared_error, r2_score, mean_absolute_error, 
                             root_mean_squared_error, confusion_matrix, f1_score, 
                             accuracy_score, recall_score, precision_score, classification_report)
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures


from sklearn.compose import ColumnTransformer


In [17]:
hr = pd.read_csv('HR_comma_sep.csv')
hr

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,Department,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.10,0.77,6,247,4,0,1,0,sales,low
3,0.92,0.85,5,259,5,0,1,0,sales,low
4,0.89,1.00,5,224,5,0,1,0,sales,low
...,...,...,...,...,...,...,...,...,...,...
14990,0.40,0.57,2,151,3,0,1,0,support,low
14991,0.37,0.48,2,160,3,0,1,0,support,low
14992,0.37,0.53,2,143,3,0,1,0,support,low
14993,0.11,0.96,6,280,4,0,1,0,support,low


In [18]:
X = hr.drop('left',axis=1)
y=hr['left']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=26)

In [19]:
from sklearn.preprocessing import OneHotEncoder

# Define the encoder object
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

ohe = ColumnTransformer(
    transformers=[
        ("OHE", encoder, make_column_selector(dtype_include=object)) # Added the encoder object here
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
).set_output(transform="pandas")

# Use X_train instead of 'hr' to prevent data leakage from the test set
X_trn_ohe = ohe.fit_transform(X_train)
X_tst_ohe = ohe.transform(X_test)


In [4]:
from sklearn.preprocessing import StandardScaler

# Improved Preprocessing: Scale the numbers so solvers work better
ohe = ColumnTransformer(transformers=[
    ("OHE", OneHotEncoder(handle_unknown='ignore', sparse_output=False), make_column_selector(dtype_include=object)),
    ("Scale", StandardScaler(), make_column_selector(dtype_exclude=object))
], verbose_feature_names_out=False).set_output(transform="pandas")

X_trn_ohe = ohe.fit_transform(X_train)
X_tst_ohe = ohe.transform(X_test)

# Now run your loop
solvers = ['lbfgs', 'newton-cg', 'newton-cholesky', 'sag', 'saga']
Cs = np.linspace(0.001, 5, 20)
scores = []

for s in solvers:
    for c in Cs:
        lr = LogisticRegression(solver=s, C=c, max_iter=1000)
        lr.fit(X_trn_ohe, y_train)
        # Use .score() as a cleaner way to get accuracy
        score = lr.score(X_tst_ohe, y_test) 
        scores.append({'solver': s, 'C': c, 'accuracy': score})

# Convert to DataFrame to easily find the best result
import pandas as pd
results_df = pd.DataFrame(scores)
print(results_df.sort_values(by='accuracy', ascending=False).head())

NameError: name 'make_column_selector' is not defined

In [ ]:
lr=LinearRegression()
lr.fit(X_trn_ohe,y_train)
y_pred =lr.predict(X_trn_ohe)
print("R2 =",r2_score(y_test,y_pred))

In [ ]:
bm=LinearRegression()
X_ohe=ohe.fit_transform(X)
bm.fit(X_ohe,y)
X_ohe.columns

In [ ]:
tst=pd.read_csv("tst_hr.csv")
tst

In [ ]:
bm=LinearRegression()
X_ohe=ohe.fit_transform(X)
bm.fit(X_ohe,y)
X_ohe.columns